# 🤖 Agentic Research Agent (Agentic AI Portfolio)

A **research agent** that plans, searches, reads sources, collects evidence, detects its own information gaps, searches again if needed, and synthesizes a final sourced report.

```
User Goal → Planning → Search → Read Sources → Collect Evidence →
Identify Information Gaps → Additional Search (if needed) →
Synthesize → Final Research Report
```

**Runs 100% in Google Colab.** Uses only a free-tier **Google Gemini** API key (no OpenAI key needed) and free web search (`ddgs`). No LangChain / LangGraph / CrewAI / AutoGen — the agent loop below is implemented by hand so the mechanics of planning, tool use, state, and gap-detection are fully visible.

## Before you run this notebook
1. Get a free Gemini API key: https://aistudio.google.com/apikey
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar.
3. Add a secret named exactly `GEMINI_API_KEY` and paste your key as the value. Turn on "Notebook access".
4. Run the cells top to bottom (`Runtime → Run all` works too).

This notebook will also generate a complete, GitHub-ready project folder (`01-research-agent/`) and let you download it as a ZIP at the end.

In [6]:
# Cell 2 — Install dependencies
# Lightweight stack only: Gemini SDK, free search, and a basic HTML parser.
!pip install -q -U google-genai ddgs requests beautifulsoup4
print("Dependencies installed.")

Dependencies installed.


In [7]:
# Cell 3 — Import libraries
import os
import re
import json
import time
import logging
import zipfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import requests
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("research_agent")

print("Libraries imported.")

Libraries imported.


In [8]:
# Cell 4 — Load Gemini API key from Colab Secrets
# NEVER paste your API key directly into a cell. Use Colab Secrets instead:
#   left sidebar -> key icon (Secrets) -> add secret named GEMINI_API_KEY

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it under Colab Secrets (key icon in the "
        "left sidebar), name it exactly 'GEMINI_API_KEY', and enable notebook access."
    )

print("Gemini API key loaded from Colab Secrets (value hidden).")

Gemini API key loaded from Colab Secrets (value hidden).


In [9]:
# Cell 5 — Configuration
# This also WRITES config.py to disk so the project is GitHub-ready from the start.

Path("config.py").write_text(r'''"""
config.py
---------
Central configuration for the Agentic Research Agent.

Everything that controls cost, behaviour, or model choice lives here so the
rest of the codebase never hard-codes a magic number or a model name.

Notes on the Gemini model:
    Gemini model availability changes over time and can differ by account
    and region. GEMINI_MODEL is deliberately NOT hard-coded into the logic
    anywhere else in this project -- it is read once, here, from an
    environment variable (with a sensible fallback). If your account does
    not have access to the default model below, either:
        1. Set the environment variable GEMINI_MODEL to a model you do
           have access to, or
        2. Edit DEFAULT_GEMINI_MODEL in this file.
    A good way to check which models your key can use is to run:
        for m in client.models.list():
            print(m.name)
"""

import os
from dataclasses import dataclass, field


# ---------------------------------------------------------------------------
# Model configuration
# ---------------------------------------------------------------------------
# This default is current as of project creation but Google regularly ships
# new Gemini model names and retires old ones. Change the env var, not this
# file, if you need a different model long-term.
DEFAULT_GEMINI_MODEL = "gemini-3.5-flash"


# ---------------------------------------------------------------------------
# Free-tier / cost safety limits
# ---------------------------------------------------------------------------
DEFAULT_MAX_AGENT_STEPS = 5      # hard ceiling on the agent loop
DEFAULT_MAX_SEARCH_RESULTS = 5   # results returned per web search call
DEFAULT_MAX_SOURCES = 8          # total sources kept across the whole run
DEFAULT_MAX_PAGE_CHARS = 12000   # characters kept per fetched webpage
DEFAULT_REQUEST_TIMEOUT = 10     # seconds, for webpage fetches


@dataclass
class Config:
    """
    Runtime configuration for the Research Agent.

    GEMINI_API_KEY is intentionally required with no default -- the agent
    should fail fast and clearly if it isn't configured, rather than
    silently doing nothing.
    """

    gemini_api_key: str
    gemini_model: str = field(
        default_factory=lambda: os.environ.get("GEMINI_MODEL", DEFAULT_GEMINI_MODEL)
    )
    max_agent_steps: int = field(
        default_factory=lambda: int(
            os.environ.get("MAX_AGENT_STEPS", DEFAULT_MAX_AGENT_STEPS)
        )
    )
    max_search_results: int = field(
        default_factory=lambda: int(
            os.environ.get("MAX_SEARCH_RESULTS", DEFAULT_MAX_SEARCH_RESULTS)
        )
    )
    max_sources: int = field(
        default_factory=lambda: int(os.environ.get("MAX_SOURCES", DEFAULT_MAX_SOURCES))
    )
    max_page_chars: int = field(
        default_factory=lambda: int(
            os.environ.get("MAX_PAGE_CHARS", DEFAULT_MAX_PAGE_CHARS)
        )
    )
    request_timeout: int = field(
        default_factory=lambda: int(
            os.environ.get("REQUEST_TIMEOUT", DEFAULT_REQUEST_TIMEOUT)
        )
    )

    def __post_init__(self):
        if not self.gemini_api_key or not str(self.gemini_api_key).strip():
            raise ValueError(
                "GEMINI_API_KEY is missing or empty. In Google Colab, add it "
                "under the Secrets (key) panel with the name 'GEMINI_API_KEY' "
                "and enable notebook access for it. Never paste the key "
                "directly into a cell."
            )

    def summary(self) -> str:
        """Human readable, secret-free summary for logging."""
        return (
            f"model={self.gemini_model} | "
            f"max_agent_steps={self.max_agent_steps} | "
            f"max_search_results={self.max_search_results} | "
            f"max_sources={self.max_sources} | "
            f"max_page_chars={self.max_page_chars}"
        )


def load_config_from_env(gemini_api_key: str) -> Config:
    """
    Build a Config using an explicitly-passed API key (e.g. loaded from
    Colab Secrets by the caller) plus whatever overrides are present in
    the environment.
    """
    return Config(gemini_api_key=gemini_api_key)
''')

from config import Config

CONFIG = Config(gemini_api_key=GEMINI_API_KEY)
print("Config loaded:", CONFIG.summary())

Config loaded: model=gemini-3.5-flash | max_agent_steps=5 | max_search_results=5 | max_sources=8 | max_page_chars=12000


In [10]:
# Cell 6 — Gemini wrapper
# Writes llm.py: the ONLY module that talks to Gemini directly.

Path("llm.py").write_text(r'''"""
llm.py
------
Thin, reusable wrapper around the Gemini API (google-genai SDK).

This is intentionally the ONLY file that talks to Gemini directly. Every
other module calls GeminiLLM.generate(...) so that:
    - the model name is configurable in one place (config.py)
    - error handling / retries live in one place
    - later portfolio projects (coding agent, sales agent, recruiting
      agent) can reuse this exact class unchanged
"""

import json
import logging
import re
import time
from typing import Optional

logger = logging.getLogger("research_agent.llm")


class GeminiError(RuntimeError):
    """Raised for any Gemini call that fails after retries."""


class GeminiLLM:
    """
    Minimal wrapper around google-genai's client.

    Usage:
        llm = GeminiLLM(api_key=GEMINI_API_KEY, model=GEMINI_MODEL)
        text = llm.generate("Write a haiku about oceans")
    """

    def __init__(self, api_key: str, model: str, max_retries: int = 2, retry_delay: float = 2.0):
        if not api_key:
            raise GeminiError("Missing Gemini API key.")

        try:
            from google import genai
        except ImportError as exc:
            raise GeminiError(
                "The 'google-genai' package is not installed. "
                "Run: pip install -q -U google-genai"
            ) from exc

        self._genai = genai
        self.model = model
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.call_count = 0  # simple usage counter for free-tier awareness

        # Client reads GEMINI_API_KEY itself, but we pass it explicitly so
        # the key never needs to live in os.environ.
        self.client = genai.Client(api_key=api_key)

    def generate(self, prompt: str, temperature: float = 0.4, max_output_tokens: int = 2048) -> str:
        """
        Send a single prompt to Gemini and return the text response.
        Raises GeminiError on unrecoverable failure (caller decides how to
        degrade gracefully).
        """
        last_error: Optional[Exception] = None

        for attempt in range(1, self.max_retries + 2):  # e.g. 1 try + N retries
            try:
                self.call_count += 1
                logger.info("Gemini call #%d (attempt %d) -> model=%s", self.call_count, attempt, self.model)

                from google.genai import types

                response = self.client.models.generate_content(
                    model=self.model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=temperature,
                        max_output_tokens=max_output_tokens,
                    ),
                )

                text = getattr(response, "text", None)
                if text is None or text.strip() == "":
                    raise GeminiError("Gemini returned an empty response.")
                return text

            except Exception as exc:  # broad: SDK raises several exception types
                last_error = exc
                message = str(exc).lower()

                # Don't retry on things a retry can't fix.
                non_retryable = ("api key" in message or "permission" in message or "not found" in message)
                if non_retryable or attempt == self.max_retries + 1:
                    break

                logger.warning(
                    "Gemini call failed (attempt %d/%d): %s -- retrying in %.1fs",
                    attempt, self.max_retries + 1, exc, self.retry_delay,
                )
                time.sleep(self.retry_delay)

        raise GeminiError(f"Gemini call failed after retries: {last_error}")


def safe_json_parse(raw_text: str) -> Optional[dict]:
    """
    Best-effort extraction of a JSON object from an LLM response.

    LLMs frequently wrap JSON in markdown fences or add stray prose before
    or after the object. This function tries several increasingly lenient
    strategies and returns None (never raises) if all of them fail, so a
    single malformed response can never crash the agent.
    """
    if not raw_text:
        return None

    candidates = []

    # Strategy 1: the whole string, as-is.
    candidates.append(raw_text.strip())

    # Strategy 2: strip ```json ... ``` or ``` ... ``` fences.
    fence_match = re.search(r"```(?:json)?\s*(.*?)```", raw_text, re.DOTALL)
    if fence_match:
        candidates.append(fence_match.group(1).strip())

    # Strategy 3: grab the substring between the first '{' and the last '}'.
    first_brace = raw_text.find("{")
    last_brace = raw_text.rfind("}")
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        candidates.append(raw_text[first_brace : last_brace + 1])

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except (json.JSONDecodeError, TypeError):
            continue

    logger.warning("safe_json_parse: could not parse JSON from model output.")
    return None
''')

from llm import GeminiLLM, GeminiError, safe_json_parse

LLM = GeminiLLM(api_key=GEMINI_API_KEY, model=CONFIG.gemini_model)

# Smoke test (uses 1 Gemini call)
try:
    test_reply = LLM.generate("Reply with the single word: OK")
    print("Gemini connectivity check:", test_reply.strip()[:50])
except GeminiError as e:
    print("Gemini call failed:", e)
    print("If this mentions the model name, try changing CONFIG.gemini_model / GEMINI_MODEL.")

Gemini connectivity check: OK


In [11]:
# Cell 7 — Tool implementations
# Writes tools.py: web_search, fetch_webpage, extract_evidence.

Path("tools.py").write_text(r'''"""
tools.py
--------
The agent's tool system: web search, webpage fetching, and evidence
extraction. These are plain Python functions/classes -- no framework,
no tool-calling protocol -- because the agent loop in research_agent.py
decides explicitly which tool to call and when.

Tools implemented:
    1. web_search(query)       -> list[dict]  (title, url, snippet)
    2. fetch_webpage(url)      -> str          (cleaned text)
    3. extract_evidence(...)   -> dict         (structured evidence)

All tools fail soft: on error they return an empty result plus a logged
warning, never an unhandled exception, so one bad source never kills a
research run.
"""

import logging
import re
from typing import List, Dict, Optional

import requests
from bs4 import BeautifulSoup

logger = logging.getLogger("research_agent.tools")

USER_AGENT = (
    "Mozilla/5.0 (compatible; AgenticResearchAgent/1.0; "
    "+https://github.com/) research-bot"
)

TRUSTED_DOMAIN_HINTS = (
    ".gov", ".edu", "wikipedia.org", "arxiv.org", "nature.com",
    "acm.org", "ieee.org", "who.int", "un.org", "nih.gov",
    "docs.", "developer.", ".org",
)


def _looks_reliable(url: str) -> bool:
    """Cheap heuristic used only for ranking/sorting, never for filtering out."""
    url_lower = (url or "").lower()
    return any(hint in url_lower for hint in TRUSTED_DOMAIN_HINTS)


# ---------------------------------------------------------------------------
# Tool 1: Web Search
# ---------------------------------------------------------------------------
def web_search(query: str, max_results: int = 5) -> List[Dict]:
    """
    Free web search using the `ddgs` package (DuckDuckGo search wrapper).

    Returns a list of dicts: {"title": ..., "url": ..., "snippet": ...}
    Returns an empty list (never raises) if the search fails or the
    dependency is unavailable.
    """
    query = (query or "").strip()
    if not query:
        return []

    try:
        from ddgs import DDGS
    except ImportError:
        logger.error("The 'ddgs' package is not installed. Run: pip install -q ddgs")
        return []

    results: List[Dict] = []
    try:
        with DDGS() as ddgs:
            for item in ddgs.text(query, max_results=max_results):
                results.append(
                    {
                        "title": item.get("title", "").strip(),
                        "url": item.get("href") or item.get("url", ""),
                        "snippet": item.get("body", "").strip(),
                    }
                )
    except Exception as exc:
        logger.warning("web_search failed for query %r: %s", query, exc)
        return []

    # Prefer results that look like reliable sources, without discarding others.
    results.sort(key=lambda r: _looks_reliable(r.get("url", "")), reverse=True)
    return results[:max_results]


# ---------------------------------------------------------------------------
# Tool 2: Webpage Fetcher
# ---------------------------------------------------------------------------
def fetch_webpage(url: str, max_chars: int = 12000, timeout: int = 10) -> str:
    """
    Fetch a URL and return cleaned, readable text (scripts/styles/nav
    stripped). Truncated to max_chars to keep Gemini prompts small.

    Returns "" (never raises) on any failure -- invalid URL, timeout,
    non-HTML content, etc.
    """
    url = (url or "").strip()
    if not url.lower().startswith(("http://", "https://")):
        logger.warning("fetch_webpage: invalid URL skipped: %r", url)
        return ""

    try:
        response = requests.get(
            url,
            headers={"User-Agent": USER_AGENT},
            timeout=timeout,
        )
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        logger.warning("fetch_webpage failed for %s: %s", url, exc)
        return ""

    content_type = response.headers.get("Content-Type", "")
    if "html" not in content_type and "text" not in content_type:
        logger.warning("fetch_webpage: skipping non-HTML content at %s (%s)", url, content_type)
        return ""

    try:
        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "header", "footer", "form", "noscript", "svg", "aside"]):
            tag.decompose()

        text = soup.get_text(separator="\n")
        # Collapse excessive whitespace/blank lines left behind by decompose().
        lines = [line.strip() for line in text.splitlines()]
        text = "\n".join(line for line in lines if line)
        text = re.sub(r"\n{3,}", "\n\n", text)

        return text[:max_chars]
    except Exception as exc:
        logger.warning("fetch_webpage: extraction failed for %s: %s", url, exc)
        return ""


# ---------------------------------------------------------------------------
# Tool 3: Source Extractor / Evidence Collector
# ---------------------------------------------------------------------------
def extract_evidence(
    source_title: str,
    source_url: str,
    page_text: str,
    query_context: str,
    max_key_points: int = 5,
    max_point_chars: int = 240,
) -> Optional[Dict]:
    """
    Convert raw page text into a compact, structured evidence record
    WITHOUT calling the LLM (kept purely heuristic to save Gemini quota).

    This is a lightweight, deterministic extractor: it splits the page
    into candidate sentences/paragraphs and keeps the ones most likely to
    be informative (reasonable length, contains some overlap with the
    query context). The LLM analyst step later decides what this evidence
    actually means -- this tool's job is just to shrink noisy HTML into a
    few candidate facts.

    Returns None if no usable text was found.
    """
    if not page_text or not page_text.strip():
        return None

    # Split into paragraph-like chunks.
    chunks = [c.strip() for c in re.split(r"\n+", page_text) if c.strip()]
    # Keep chunks of reasonable "sentence-like" length.
    candidates = [c for c in chunks if 40 <= len(c) <= 500]

    if not candidates:
        candidates = [c for c in chunks if len(c) > 20][:max_key_points]

    query_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", query_context or "")}

    def relevance_score(chunk: str) -> int:
        chunk_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", chunk)}
        return len(query_terms & chunk_terms)

    candidates.sort(key=relevance_score, reverse=True)

    key_points = []
    for chunk in candidates[: max_key_points * 3]:
        snippet = chunk if len(chunk) <= max_point_chars else chunk[:max_point_chars].rsplit(" ", 1)[0] + "..."
        if snippet not in key_points:
            key_points.append(snippet)
        if len(key_points) >= max_key_points:
            break

    if not key_points:
        return None

    return {
        "source": source_url,
        "title": source_title or source_url,
        "key_points": key_points,
        "relevance": "high" if relevance_score(" ".join(key_points)) > 0 else "unknown",
    }
''')

from tools import web_search, fetch_webpage, extract_evidence

# Smoke test (free, no Gemini call)
_results = web_search("agentic AI systems", max_results=3)
print(f"web_search smoke test: {len(_results)} result(s)")
for r in _results:
    print(" -", r["title"][:70], "|", r["url"])

web_search smoke test: 3 result(s)
 - Agentic AI, explained | MIT Sloan | https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained
 - What is Agentic AI? | IBM | https://www.ibm.com/think/topics/agentic-ai
 - What is Agentic AI? - Agentic AI Explained - AWS | https://aws.amazon.com/what-is/agentic-ai/


In [12]:
# Cells 8–12 — Agent state, Planner, Research Analyst, Agent loop, Final report generator
# These all live together in research_agent.py for cohesion; this single cell
# writes the file and imports every piece so later cells can use it directly.
#
#   Cell 8  -> ResearchState (agent state)
#   Cell 9  -> PLANNER_PROMPT (planner)
#   Cell 10 -> ANALYST_PROMPT (research analyst / gap detection)
#   Cell 11 -> ResearchAgent.run() (the agent loop)
#   Cell 12 -> ResearchAgent._synthesize() + SYNTHESIS_PROMPT (final report generator)

Path("research_agent.py").write_text(r'''"""
research_agent.py
------------------
The core agentic system: state, prompt templates, and the agent loop that
ties the LLM (llm.py) and the tools (tools.py) together.

This file deliberately implements the agent loop BY HAND (no LangChain /
LangGraph / CrewAI / AutoGen) to demonstrate understanding of the
fundamentals of agentic AI:

    inspect_state -> decide_next_action -> execute_tool ->
    save_observation -> evaluate_information_gaps -> ... -> synthesize

The loop is bounded by config.max_agent_steps so it can never run away on
a free-tier API key.
"""

import logging
from dataclasses import dataclass, field
from typing import List, Dict, Optional

from config import Config
from llm import GeminiLLM, GeminiError, safe_json_parse
from tools import web_search, fetch_webpage, extract_evidence

logger = logging.getLogger("research_agent.core")


# ---------------------------------------------------------------------------
# Agent State
# ---------------------------------------------------------------------------
@dataclass
class ResearchState:
    """
    Explicit, persistent state for a single research run.

    Keeping this as one object (rather than scattering variables through
    the loop) is what makes this an *agent* with memory of its own
    progress, rather than a single stateless prompt->response call.
    """

    question: str
    plan: List[str] = field(default_factory=list)
    searches: List[Dict] = field(default_factory=list)     # {"query": ..., "result_count": ...}
    sources: List[Dict] = field(default_factory=list)       # raw search hits, deduped by URL
    evidence: List[Dict] = field(default_factory=list)       # structured evidence records
    gaps: List[str] = field(default_factory=list)            # open sub-questions still unanswered
    analyst_notes: List[Dict] = field(default_factory=list)  # raw analyst JSON per round
    final_report: str = ""
    steps: int = 0
    llm_calls: int = 0
    search_calls: int = 0

    def seen_urls(self) -> set:
        return {s["url"] for s in self.sources if s.get("url")}


# ---------------------------------------------------------------------------
# Prompt Templates
# ---------------------------------------------------------------------------
PLANNER_PROMPT = """You are a research planner. Break the user's research question
into 2 to 5 focused, non-overlapping sub-questions that, together, would let
someone write a thorough answer. Prefer sub-questions that map naturally to
things a web search could answer.

Research question: {question}

Respond with ONLY valid JSON in this exact shape, no extra commentary:
{{
  "sub_questions": ["...", "...", "..."]
}}
"""

ANALYST_PROMPT = """You are a careful research analyst. Below is evidence collected
so far for the research question: {question}

Sub-questions being investigated:
{sub_questions}

Evidence collected so far:
{evidence_block}

Analyze this evidence and respond with ONLY valid JSON in this exact shape:
{{
  "key_findings": ["...", "..."],
  "conflicting_information": ["..."],
  "missing_information": ["..."],
  "needs_more_research": true,
  "suggested_followup_queries": ["...", "..."]
}}

Rules:
- "needs_more_research" should be false only if the evidence already
  reasonably covers the sub-questions.
- "suggested_followup_queries" should be empty if needs_more_research is false.
- Only claim a conflict if sources genuinely disagree.
- Do not invent findings that are not supported by the evidence above.
"""

SYNTHESIS_PROMPT = """You are a research analyst writing a final report. Use ONLY the
evidence provided below -- do not invent facts, statistics, or sources that
are not present in it.

Research question: {question}

Research plan (sub-questions investigated):
{sub_questions}

All evidence collected (each item includes its source URL):
{evidence_block}

Analyst notes from the investigation:
{analyst_summary}

Write a final report with EXACTLY these sections, using Markdown headings:

## Executive Summary
A short (3-5 sentence) overview of the answer to the research question.

## Key Findings
A bulleted list of the most important, well-supported findings.

## Detailed Analysis
A few paragraphs synthesizing the evidence into a coherent narrative,
noting agreement/disagreement between sources where relevant.

## Evidence
A bulleted list connecting specific claims to specific sources (cite by
URL or title).

## Limitations
Note gaps in the research, remaining uncertainty, and the general caveat
that web search results are not guaranteed to be complete or accurate.

## Sources
A numbered list of every source URL actually used above. Do not invent
URLs that were not provided in the evidence.
"""


# ---------------------------------------------------------------------------
# Helper: format evidence for prompts (keeps prompts compact)
# ---------------------------------------------------------------------------
def _format_evidence_block(evidence: List[Dict]) -> str:
    if not evidence:
        return "(no evidence collected yet)"
    lines = []
    for i, item in enumerate(evidence, start=1):
        points = "; ".join(item.get("key_points", []))
        lines.append(f"[{i}] {item.get('title', 'Untitled')} ({item.get('source', 'no-url')})\n    {points}")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# The Research Agent
# ---------------------------------------------------------------------------
class ResearchAgent:
    """
    Orchestrates the full agentic research workflow:

        Plan -> Search -> Read -> Collect Evidence -> Identify Gaps ->
        (Search again if needed) -> Synthesize Final Report

    The agent decides for itself, based on the analyst's assessment,
    whether another round of searching is warranted -- bounded by
    config.max_agent_steps so it can never loop forever.
    """

    def __init__(self, config: Config, llm: Optional[GeminiLLM] = None):
        self.config = config
        self.llm = llm or GeminiLLM(api_key=config.gemini_api_key, model=config.gemini_model)

    # -- Step: Planning ----------------------------------------------------
    def _plan(self, state: ResearchState, log) -> None:
        log("PLAN", f"Planning research for: {state.question}")
        prompt = PLANNER_PROMPT.format(question=state.question)
        try:
            raw = self.llm.generate(prompt)
            state.llm_calls += 1
            parsed = safe_json_parse(raw)
            sub_qs = (parsed or {}).get("sub_questions") or []
            sub_qs = [q for q in sub_qs if isinstance(q, str) and q.strip()][:5]
        except GeminiError as exc:
            log("WARN", f"Planner LLM call failed ({exc}); falling back to the raw question.")
            sub_qs = []

        if not sub_qs:
            sub_qs = [state.question]

        state.plan = sub_qs
        log("OK", f"Research plan created ({len(sub_qs)} sub-questions).")

    # -- Step: Search + Read + Collect Evidence -----------------------------
    def _research_round(self, state: ResearchState, queries: List[str], log) -> None:
        for query in queries:
            if len(state.sources) >= self.config.max_sources:
                log("INFO", "Source limit reached; skipping further searches this round.")
                break

            log("SEARCH", f'"{query}"')
            results = web_search(query, max_results=self.config.max_search_results)
            state.search_calls += 1
            state.searches.append({"query": query, "result_count": len(results)})

            if not results:
                log("WARN", "No results found for this query.")
                continue
            log("OK", f"{len(results)} results found")

            for result in results:
                if len(state.sources) >= self.config.max_sources:
                    break
                url = result.get("url")
                if not url or url in state.seen_urls():
                    continue

                state.sources.append(result)

                log("READ", url)
                page_text = fetch_webpage(url, max_chars=self.config.max_page_chars, timeout=self.config.request_timeout)
                if not page_text:
                    # Fall back to the search snippet so the source isn't wasted.
                    page_text = result.get("snippet", "")

                evidence = extract_evidence(
                    source_title=result.get("title", ""),
                    source_url=url,
                    page_text=page_text,
                    query_context=query,
                )
                if evidence:
                    state.evidence.append(evidence)
                    log("OK", "Evidence collected")
                else:
                    log("WARN", "No usable evidence extracted from this source.")

    # -- Step: Analyze gaps --------------------------------------------------
    def _analyze(self, state: ResearchState, log) -> Dict:
        log("ANALYZE", "Analyzing research gaps...")
        prompt = ANALYST_PROMPT.format(
            question=state.question,
            sub_questions="\n".join(f"- {q}" for q in state.plan),
            evidence_block=_format_evidence_block(state.evidence),
        )
        try:
            raw = self.llm.generate(prompt)
            state.llm_calls += 1
            parsed = safe_json_parse(raw) or {}
        except GeminiError as exc:
            log("WARN", f"Analyst LLM call failed ({exc}); assuming research is sufficient.")
            parsed = {}

        parsed.setdefault("key_findings", [])
        parsed.setdefault("conflicting_information", [])
        parsed.setdefault("missing_information", [])
        parsed.setdefault("needs_more_research", False)
        parsed.setdefault("suggested_followup_queries", [])

        state.analyst_notes.append(parsed)
        state.gaps = parsed.get("missing_information", [])

        if parsed["needs_more_research"]:
            log("OK", "Additional research required")
        else:
            log("OK", "Evidence looks sufficient")

        return parsed

    # -- Step: Synthesize final report ---------------------------------------
    def _synthesize(self, state: ResearchState, log) -> None:
        log("SYNTHESIZE", "Synthesizing final report...")

        analyst_summary_lines = []
        for i, note in enumerate(state.analyst_notes, start=1):
            analyst_summary_lines.append(
                f"Round {i}: findings={note.get('key_findings')}; "
                f"conflicts={note.get('conflicting_information')}; "
                f"missing={note.get('missing_information')}"
            )
        analyst_summary = "\n".join(analyst_summary_lines) or "(no analyst notes)"

        prompt = SYNTHESIS_PROMPT.format(
            question=state.question,
            sub_questions="\n".join(f"- {q}" for q in state.plan),
            evidence_block=_format_evidence_block(state.evidence),
            analyst_summary=analyst_summary,
        )

        try:
            report = self.llm.generate(prompt, max_output_tokens=3072)
            state.llm_calls += 1
        except GeminiError as exc:
            log("WARN", f"Synthesis LLM call failed ({exc}); building a fallback report.")
            report = self._fallback_report(state)

        state.final_report = report

    def _fallback_report(self, state: ResearchState) -> str:
        """A deterministic, no-LLM report used only if Gemini is unavailable
        at the synthesis step, so the user always gets *something* back."""
        sources = "\n".join(f"- {s.get('title', 'Untitled')}: {s.get('url', '')}" for s in state.sources)
        evidence = _format_evidence_block(state.evidence)
        return (
            f"## Executive Summary\n"
            f"Automated synthesis was unavailable, so this is a raw evidence dump "
            f"for: {state.question}\n\n"
            f"## Evidence\n{evidence}\n\n"
            f"## Limitations\nThe final synthesis LLM call failed; this report is "
            f"unprocessed evidence only. Web search results are not guaranteed to "
            f"be complete or accurate.\n\n"
            f"## Sources\n{sources}\n"
        )

    # -- The Agent Loop -------------------------------------------------------
    def run(self, question: str, verbose: bool = True) -> ResearchState:
        """
        Execute the full agentic research workflow for a single question
        and return the populated ResearchState.
        """
        state = ResearchState(question=question)

        def log(tag: str, message: str) -> None:
            if verbose:
                icon = {
                    "PLAN": "\U0001F9E0", "SEARCH": "\U0001F50E", "READ": "\U0001F4C4",
                    "ANALYZE": "\U0001F9E0", "SYNTHESIZE": "\U0001F9E0",
                    "OK": "\u2713", "WARN": "\u26A0", "INFO": "\u2139",
                }.get(tag, "-")
                print(f"{icon} {message}")
            logger.info("[%s] %s", tag, message)

        if verbose:
            print("=" * 50)
            print("\U0001F916 AGENTIC RESEARCH AGENT")
            print("=" * 50)
            print(f"\n\U0001F3AF Goal:\n{question}\n")

        # 1) Plan
        state.steps += 1
        self._plan(state, log)

        # 2) Initial research round, based on the plan
        state.steps += 1
        if verbose:
            print()
        self._research_round(state, state.plan, log)

        # 3) Analyze / iterate while budget remains
        while state.steps < self.config.max_agent_steps:
            state.steps += 1
            if verbose:
                print()
            analysis = self._analyze(state, log)

            if not analysis.get("needs_more_research"):
                break

            followups = [q for q in analysis.get("suggested_followup_queries", []) if isinstance(q, str) and q.strip()]
            followups = [q for q in followups if q not in [s["query"] for s in state.searches]][:2]

            if not followups or len(state.sources) >= self.config.max_sources:
                log("INFO", "No further useful searches available within budget; moving to synthesis.")
                break

            if verbose:
                print()
            self._research_round(state, followups, log)

        # 4) Final synthesis (always runs, even if the loop hit the step cap)
        if verbose:
            print()
        self._synthesize(state, log)

        if verbose:
            print("\n" + "=" * 50)
            print("\U0001F4CA FINAL RESEARCH REPORT")
            print("=" * 50 + "\n")
            print(state.final_report)
            print("\n" + "-" * 50)
            print(
                f"Run summary: steps={state.steps} | llm_calls={state.llm_calls} | "
                f"search_calls={state.search_calls} | sources={len(state.sources)}"
            )
            print("-" * 50)

        return state


def build_agent(gemini_api_key: str) -> ResearchAgent:
    """Convenience factory used by the notebook / GitHub demo entry point."""
    config = Config(gemini_api_key=gemini_api_key)
    return ResearchAgent(config=config)
''')

from research_agent import (
    ResearchState, PLANNER_PROMPT, ANALYST_PROMPT, SYNTHESIS_PROMPT,
    ResearchAgent, build_agent,
)

print("research_agent.py written and loaded.")
print("- Agent state:      ResearchState")
print("- Planner prompt:    PLANNER_PROMPT")
print("- Analyst prompt:    ANALYST_PROMPT")
print("- Synthesis prompt:  SYNTHESIS_PROMPT")
print("- Agent loop:        ResearchAgent.run()")

research_agent.py written and loaded.
- Agent state:      ResearchState
- Planner prompt:    PLANNER_PROMPT
- Analyst prompt:    ANALYST_PROMPT
- Synthesis prompt:  SYNTHESIS_PROMPT
- Agent loop:        ResearchAgent.run()


### Cell 9 — Planner (reference)
Turns the user's question into 2–5 focused sub-questions. One Gemini call, structured JSON output (parsed safely — malformed JSON never crashes the run).

### Cell 10 — Research Analyst (reference)
Inspects collected evidence and returns structured JSON: key findings, conflicting information, missing information, and whether more research is needed. This is what makes the loop *agentic* — the agent decides for itself whether to search again.

### Cell 11 — Agent loop (reference)
`ResearchAgent.run()` implements: `plan → research_round → (analyze → research_round)* → synthesize`, bounded by `CONFIG.max_agent_steps`.

### Cell 12 — Final report generator (reference)
One last Gemini call turns all evidence + analyst notes into a Markdown report with Executive Summary, Key Findings, Detailed Analysis, Evidence, Limitations, and Sources — constrained to only cite sources actually collected.

In [13]:
# Cell 13 — Build the complete Research Agent
agent = ResearchAgent(config=CONFIG, llm=LLM)
print("Research Agent ready.")
print(f"Model: {CONFIG.gemini_model} | Max steps: {CONFIG.max_agent_steps} | Max sources: {CONFIG.max_sources}")

Research Agent ready.
Model: gemini-3.5-flash | Max steps: 5 | Max sources: 8


In [14]:
# Cell 14 — Run demo research
DEMO_QUESTIONS = [
    "What are the main components of modern AI agent architectures?",
    "Compare RAG-based systems with long-context LLM approaches.",
    "What are the major challenges in deploying autonomous AI agents?",
]

# Run the first demo question end-to-end (uses ~2-4 Gemini calls).
demo_state = agent.run(DEMO_QUESTIONS[0])

🤖 AGENTIC RESEARCH AGENT

🎯 Goal:
What are the main components of modern AI agent architectures?

🧠 Planning research for: What are the main components of modern AI agent architectures?
✓ Research plan created (4 sub-questions).

🔎 "What are the primary planning and task decomposition techniques used in modern AI agent architectures?"
✓ 5 results found
📄 https://arxiv.org/html/2601.01743v1


✓ Evidence collected
📄 https://apxml.com/courses/agentic-llm-memory-architectures/chapter-4-complex-planning-tool-integration/task-decomposition-strategies
✓ Evidence collected
📄 https://sparkco.ai/blog/deep-dive-into-agent-task-decomposition-techniques
✓ Evidence collected
📄 https://www.ibm.com/think/topics/ai-agent-planning
✓ Evidence collected
📄 https://neurals.ca/agents/concepts/planning/
✓ Evidence collected
🔎 "How do memory systems, including short-term and long-term memory, function within AI agents?"
✓ 5 results found
📄 https://www.geeksforgeeks.org/artificial-intelligence/ai-agent-memory/
✓ Evidence collected
📄 https://aiagentmemory.org/articles/short-term-and-long-term-memory-agentic-ai/


✓ Evidence collected
📄 https://medium.com/@kishie-tech-ai/memory-in-agentic-ai-short-term-long-term-and-episodic-memory-51548a937131
✓ Evidence collected
ℹ Source limit reached; skipping further searches this round.

🧠 Analyzing research gaps...
✓ Additional research required
ℹ No further useful searches available within budget; moving to synthesis.

🧠 Synthesizing final report...

📊 FINAL RESEARCH REPORT

## Executive Summary

Modern AI agent architectures are built upon three core integrated components: planning, memory, and tool execution. Rather than acting as simple text generators, agents function as controllers that translate user intent into actions through an execution loop of reasoning, tools, and memory. Planning and task decomposition allow agents to break complex, multi-faceted goals into manageable steps using frameworks like ReAct and techniques like Tree-of-Thought. Memory systems support these processes by maintaining a dual-structure of short-term context and long-ter

In [15]:
# Cell 15 — Run custom research
# Change this to any question you like, then run the cell.
custom_question = "What are the current best practices for evaluating LLM-based agents?"

custom_state = agent.run(custom_question)

🤖 AGENTIC RESEARCH AGENT

🎯 Goal:
What are the current best practices for evaluating LLM-based agents?

🧠 Planning research for: What are the current best practices for evaluating LLM-based agents?
✓ Research plan created (4 sub-questions).

🔎 "What are the leading industry-standard benchmarks and datasets used to evaluate LLM-based agents?"
✓ 5 results found
📄 https://arxiv.org/html/2507.21504v1
✓ Evidence collected
📄 https://www.aice-lab.org/posts/features/llm-benchmarks-complete-guide-2026/
✓ Evidence collected
📄 https://www.confident-ai.com/blog/llm-benchmarks-mmlu-hellaswag-and-beyond
✓ Evidence collected
📄 https://www.evidentlyai.com/llm-guide/llm-benchmarks
✓ Evidence collected
📄 https://www.langchain.com/resources/how-to-evaluate-llms
✓ Evidence collected
🔎 "What are the primary evaluation methodologies, such as LLM-as-a-judge, trajectory evaluation, and human-in-the-loop testing, for agentic workflows?"
✓ 5 results found
📄 https://arxiv.org/html/2508.02994v1


✓ Evidence collected
📄 https://medium.com/@vinodkrane/chapter-8-agent-evaluation-for-llms-how-to-test-tools-trajectories-and-llm-as-judge-788f6f3e0d52
✓ Evidence collected
📄 https://www.getmaxim.ai/articles/llm-as-a-judge-vs-human-in-the-loop-evaluations-a-complete-guide-for-ai-engineers/
✓ Evidence collected
ℹ Source limit reached; skipping further searches this round.

🧠 Analyzing research gaps...


✓ Evidence looks sufficient

🧠 Synthesizing final report...


⚠ Synthesis LLM call failed (Gemini call failed after retries: Gemini returned an empty response.); building a fallback report.

📊 FINAL RESEARCH REPORT

## Executive Summary
Automated synthesis was unavailable, so this is a raw evidence dump for: What are the current best practices for evaluating LLM-based agents?

## Evidence
[1] Evaluation and Benchmarking of LLM Agents: A Survey - arXiv.org (https://arxiv.org/html/2507.21504v1)
    We propose a taxonomy of LLM agent evaluation that organizes prior work by evaluation objectives (what to evaluate, such as behavior, capabilities, reliability, and safety) and evaluation process (how to evaluate, including interaction...; LLM Agents; Agent Evaluation; Evaluation Taxonomy; Agent Behavior, Benchmarks, Safety; Enterprise AI; Agents based on LLMs are autonomous or semi-autonomous systems that use LLMs to reason, plan, and act, and represent a rapidly growing frontier in artificial intelligence; We propose a two-dimensional taxonomy to organ

In [16]:
# Cell 16 — Display sources
def show_sources(state):
    print(f"Sources used for: {state.question}\n")
    if not state.sources:
        print("(no sources collected)")
        return
    for i, s in enumerate(state.sources, start=1):
        print(f"[{i}] {s.get('title', 'Untitled')}")
        print(f"    {s.get('url', '')}")

show_sources(demo_state)
print()
show_sources(custom_state)

Sources used for: What are the main components of modern AI agent architectures?

[1] AI Agent Systems: Architectures, Applications, and Evaluation
    https://arxiv.org/html/2601.01743v1
[2] LLM Agent Task Decomposition Strategies
    https://apxml.com/courses/agentic-llm-memory-architectures/chapter-4-complex-planning-tool-integration/task-decomposition-strategies
[3] Deep Dive into Agent Task Decomposition Techniques
    https://sparkco.ai/blog/deep-dive-into-agent-task-decomposition-techniques
[4] What is AI Agent Planning? | IBM
    https://www.ibm.com/think/topics/ai-agent-planning
[5] How agents plan, task decomposition · neurals
    https://neurals.ca/agents/concepts/planning/
[6] AI Agent Memory - GeeksforGeeks
    https://www.geeksforgeeks.org/artificial-intelligence/ai-agent-memory/
[7] Short Term and Long Term Memory in Agentic AI
    https://aiagentmemory.org/articles/short-term-and-long-term-memory-agentic-ai/
[8] Memory in Agentic AI: Short-Term, Long-Term, and Episodic 

In [17]:
# Cell 17 — Unit tests
# Writes tests.py and runs the unit tests (no Gemini API key required for these).

Path("tests.py").write_text(r'''"""
tests.py
--------
Lightweight tests runnable inside Colab (or plain Python) with no test
framework dependency beyond the standard library.

Split into two groups:
    - UNIT TESTS: no network, no API key required. Always run these.
    - API TESTS:  require GEMINI_API_KEY and network access. Skipped
      automatically if a key isn't available.

Run with:  python tests.py
"""

import sys
import traceback

from config import Config
from llm import safe_json_parse
from tools import extract_evidence, web_search
from research_agent import ResearchState


def _check(name, condition):
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {name}")
    return condition


# ---------------------------------------------------------------------------
# 1. Configuration loading
# ---------------------------------------------------------------------------
def test_config_loading():
    cfg = Config(gemini_api_key="dummy-key-for-tests")
    ok = True
    ok &= _check("config has a model name", bool(cfg.gemini_model))
    ok &= _check("config has positive max_agent_steps", cfg.max_agent_steps > 0)
    ok &= _check("config has positive max_search_results", cfg.max_search_results > 0)

    raised = False
    try:
        Config(gemini_api_key="")
    except ValueError:
        raised = True
    ok &= _check("config raises ValueError on empty API key", raised)
    return ok


# ---------------------------------------------------------------------------
# 2. JSON parsing
# ---------------------------------------------------------------------------
def test_json_parsing():
    ok = True
    ok &= _check("parses clean JSON", safe_json_parse('{"a": 1}') == {"a": 1})
    fenced = 'Sure, here you go:\n```json\n{"a": 2}\n```\nHope that helps!'
    ok &= _check("parses fenced JSON", (safe_json_parse(fenced) or {}).get("a") == 2)
    ok &= _check("returns None on garbage input", safe_json_parse("this is not json") is None)
    ok &= _check("returns None on empty input", safe_json_parse("") is None)
    return ok


# ---------------------------------------------------------------------------
# 3. Search tool response format (does not require network to validate shape;
#    if network is unavailable it should degrade to an empty list, not crash)
# ---------------------------------------------------------------------------
def test_search_tool_format():
    ok = True
    try:
        results = web_search("test query unit test", max_results=2)
        ok &= _check("web_search returns a list", isinstance(results, list))
        if results:
            first = results[0]
            ok &= _check("result has 'title'/'url'/'snippet' keys", all(k in first for k in ("title", "url", "snippet")))
        else:
            print("[INFO] web_search returned no results (offline sandbox or blocked network) -- shape check skipped.")
    except Exception:
        ok = False
        print("[FAIL] web_search raised an exception instead of degrading gracefully:")
        traceback.print_exc()
    return ok


# ---------------------------------------------------------------------------
# 4. Webpage extraction (offline, using a hand-built HTML string so this
#    test doesn't depend on network access)
# ---------------------------------------------------------------------------
def test_webpage_extraction():
    from bs4 import BeautifulSoup

    html = """
    <html><head><style>body{color:red}</style></head>
    <body>
      <nav>Home | About</nav>
      <script>console.log('noise')</script>
      <article><p>Agentic AI systems plan, act, and observe in loops.</p></article>
      <footer>Copyright 2026</footer>
    </body></html>
    """
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    text = soup.get_text(separator="\n")

    ok = True
    ok &= _check("script content removed", "console.log" not in text)
    ok &= _check("nav content removed", "About" not in text)
    ok &= _check("main content preserved", "plan, act, and observe" in text)
    return ok


# ---------------------------------------------------------------------------
# 5. Agent state initialization
# ---------------------------------------------------------------------------
def test_agent_state_init():
    state = ResearchState(question="What is an AI agent?")
    ok = True
    ok &= _check("question stored", state.question == "What is an AI agent?")
    ok &= _check("plan starts empty", state.plan == [])
    ok &= _check("sources start empty", state.sources == [])
    ok &= _check("evidence starts empty", state.evidence == [])
    ok &= _check("steps start at 0", state.steps == 0)
    ok &= _check("seen_urls starts empty", state.seen_urls() == set())
    return ok


# ---------------------------------------------------------------------------
# Extra: evidence extraction unit test (unit-level, no network/API)
# ---------------------------------------------------------------------------
def test_evidence_extraction():
    page_text = (
        "Modern AI agent architectures typically include a planning module, "
        "a memory or state component, and a set of callable tools.\n"
        "Short line.\n"
        "Agents differ from simple chatbots because they can take multiple "
        "autonomous steps toward a goal before responding to the user."
    )
    evidence = extract_evidence("Agent Architectures", "https://example.com/agents", page_text, "agent architecture components")
    ok = True
    ok &= _check("evidence extracted", evidence is not None)
    if evidence:
        ok &= _check("evidence has key_points", len(evidence.get("key_points", [])) > 0)
        ok &= _check("evidence retains source url", evidence.get("source") == "https://example.com/agents")
    return ok


UNIT_TESTS = [
    ("Configuration loading", test_config_loading),
    ("JSON parsing", test_json_parsing),
    ("Search tool response format", test_search_tool_format),
    ("Webpage extraction", test_webpage_extraction),
    ("Agent state initialization", test_agent_state_init),
    ("Evidence extraction", test_evidence_extraction),
]


def run_all():
    print("=" * 50)
    print("RUNNING UNIT TESTS (no Gemini API key required)")
    print("=" * 50)
    results = []
    for name, fn in UNIT_TESTS:
        print(f"\n-- {name} --")
        try:
            results.append(fn())
        except Exception:
            print(f"[FAIL] {name} raised an unexpected exception:")
            traceback.print_exc()
            results.append(False)

    passed = sum(results)
    total = len(results)
    print("\n" + "=" * 50)
    print(f"RESULT: {passed}/{total} test groups passed")
    print("=" * 50)
    return passed == total


if __name__ == "__main__":
    success = run_all()
    sys.exit(0 if success else 1)
''')

import importlib
import tests as tests_module
importlib.reload(tests_module)

tests_passed = tests_module.run_all()
print("\nAll unit tests passed!" if tests_passed else "\nSome unit tests failed — see output above.")

RUNNING UNIT TESTS (no Gemini API key required)

-- Configuration loading --
[PASS] config has a model name
[PASS] config has positive max_agent_steps
[PASS] config has positive max_search_results
[PASS] config raises ValueError on empty API key

-- JSON parsing --
[PASS] parses clean JSON
[PASS] parses fenced JSON
[PASS] returns None on garbage input
[PASS] returns None on empty input

-- Search tool response format --
[PASS] web_search returns a list
[PASS] result has 'title'/'url'/'snippet' keys

-- Webpage extraction --
[PASS] script content removed
[PASS] nav content removed
[PASS] main content preserved

-- Agent state initialization --
[PASS] question stored
[PASS] plan starts empty
[PASS] sources start empty
[PASS] evidence starts empty
[PASS] steps start at 0
[PASS] seen_urls starts empty

-- Evidence extraction --
[PASS] evidence extracted
[PASS] evidence has key_points
[PASS] evidence retains source url

RESULT: 6/6 test groups passed

All unit tests passed!


In [18]:
# Cell 18 — Generate remaining GitHub project files
# config.py, llm.py, tools.py, research_agent.py, and tests.py were already
# written to disk in Cells 5-8 and 17. This cell adds the rest of the
# GitHub-ready project: .env.example and the sample queries data file.

Path("data").mkdir(exist_ok=True)

Path(".env.example").write_text(r'''# Copy this file to .env for local (non-Colab) use and fill in your key.
# NEVER commit a real .env file with a real key to GitHub.

# Required: your Gemini API key (get one at https://aistudio.google.com/apikey)
GEMINI_API_KEY=your-gemini-api-key-here

# Optional overrides -- see config.py for defaults.
GEMINI_MODEL=gemini-3.5-flash
MAX_AGENT_STEPS=5
MAX_SEARCH_RESULTS=5
MAX_SOURCES=8
MAX_PAGE_CHARS=12000
REQUEST_TIMEOUT=10

# NOTE: In Google Colab, do NOT use this file. Instead add GEMINI_API_KEY
# under the Colab "Secrets" (key icon) panel and load it with:
#   from google.colab import userdata
#   GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
''')
Path("data/sample_queries.txt").write_text(r'''What are the main components of modern AI agent architectures?
Compare RAG-based systems with long-context LLM approaches.
What are the major challenges in deploying autonomous AI agents?
How is multi-agent orchestration different from single-agent workflows?
What are the current best practices for evaluating LLM-based agents?
''')

print("Generated: .env.example")
print("Generated: data/sample_queries.txt")

Generated: .env.example
Generated: data/sample_queries.txt


In [19]:
# Cell 19 — Generate README.md
Path("README.md").write_text(r'''# Agentic Research Agent

Project 1 of a 4-part Agentic AI portfolio (`01 Research Agent → 02 Coding Agent → 03 Sales Agent → 04 Recruiting Agent`).

A research agent that **plans, searches, reads, gathers evidence, detects its own information gaps, searches again if needed, and synthesizes a sourced report** — built from scratch in Python, runnable entirely in Google Colab on a free Gemini API key.

## Overview

Most "AI research tools" are a single call:

```
Question → LLM → Answer
```

That's not an agent — it's a prompt. This project implements an actual **agentic loop**: the system maintains state across multiple steps, decides for itself which tool to call next, evaluates whether its own evidence is sufficient, and only stops when it has enough to answer (or it hits a hard step budget).

```
User Goal → Planning → Search → Read Sources → Collect Evidence →
Identify Information Gaps → Additional Search (if needed) →
Synthesize → Final Research Report
```

No LangChain, LangGraph, CrewAI, or AutoGen — the planning/tool-use/state loop is implemented directly so the mechanics of an agent are fully visible and understandable, not hidden behind a framework.

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                         ResearchAgent                            │
│                                                                    │
│   ┌───────────┐    ┌──────────────┐    ┌────────────────────┐    │
│   │  Planner  │───▶│  Agent Loop  │───▶│  Research Analyst   │    │
│   │ (Gemini)  │    │ (state +     │◀───│      (Gemini)        │   │
│   └───────────┘    │  budget)     │    └────────────────────┘    │
│                     └──────┬───────┘                              │
│                            │                                      │
│                     ┌──────▼───────┐                              │
│                     │    Tools     │                              │
│                     │──────────────│                              │
│                     │ web_search   │ (ddgs)                       │
│                     │ fetch_webpage│ (requests + bs4)              │
│                     │ extract_     │                              │
│                     │  evidence    │                              │
│                     └──────┬───────┘                              │
│                            │                                      │
│                     ┌──────▼───────┐                              │
│                     │ ResearchState│  question, plan, searches,   │
│                     │  (dataclass) │  sources, evidence, gaps,     │
│                     │              │  final_report, steps          │
│                     └──────────────┘                              │
│                            │                                      │
│                     ┌──────▼───────┐                              │
│                     │ Synthesizer  │───▶ Final Markdown Report     │
│                     │  (Gemini)    │                              │
│                     └──────────────┘                              │
└─────────────────────────────────────────────────────────────────┘
```

## How the Agent Works

1. **Planning** — one Gemini call turns the user's question into 2–5 focused sub-questions.
2. **Search** — the agent runs `web_search()` (DuckDuckGo via `ddgs`, no paid API) for each sub-question.
3. **Read Sources** — each result URL is fetched with `fetch_webpage()` and cleaned into readable text (scripts/nav/styles stripped, truncated to a character budget).
4. **Collect Evidence** — `extract_evidence()` heuristically condenses each page into a small structured record (`source`, `title`, `key_points`, `relevance`) *without* an LLM call, so no Gemini quota is spent just to shrink HTML.
5. **Identify Gaps** — one Gemini call (the "Research Analyst") inspects all evidence collected so far and returns structured JSON: key findings, conflicts, missing information, and whether more research is needed.
6. **Additional Search (conditional)** — if the analyst says more research is needed *and* there's step/source budget left, the agent runs another search round on the analyst's suggested follow-up queries. Otherwise it moves straight to synthesis.
7. **Synthesize** — one final Gemini call turns all evidence + analyst notes into a structured Markdown report with cited sources.

This is what makes it *agentic* rather than a single prompt: the number of search rounds is not fixed in advance — the agent decides, based on its own evaluation of gaps, whether to keep researching.

## Agent Loop

The loop lives in `research_agent.py: ResearchAgent.run()`:

```
state = ResearchState(question)

plan()                                   # 1 Gemini call
research_round(state.plan)               # tools only, no LLM

while state.steps < MAX_AGENT_STEPS:
    analysis = analyze(state)            # 1 Gemini call
    if not analysis.needs_more_research:
        break
    research_round(analysis.followups)   # tools only, no LLM

synthesize(state)                        # 1 Gemini call
```

`MAX_AGENT_STEPS` (default 5) hard-caps the loop so it can never run away on a free-tier key. A typical run uses **2–4 Gemini calls total**.

## Tools

| Tool | File | Backing library | Cost |
|---|---|---|---|
| `web_search(query)` | `tools.py` | `ddgs` (DuckDuckGo) | free |
| `fetch_webpage(url)` | `tools.py` | `requests` + `beautifulsoup4` | free |
| `extract_evidence(...)` | `tools.py` | pure Python heuristics | free, no LLM |

All three tools **fail soft**: a bad URL, a timeout, an empty search, or a parsing error returns an empty result and logs a warning — it never crashes the run, so one broken source can't derail the whole research task.

## Project Structure

```
01-research-agent/
│
├── README.md              this file
├── research_agent.py       agent state, prompts, agent loop, ResearchAgent
├── tools.py                web_search, fetch_webpage, extract_evidence
├── llm.py                  GeminiLLM wrapper + safe_json_parse
├── config.py                Config dataclass, free-tier limits
├── tests.py                 unit tests (no API key needed) + notes on API tests
├── requirements.txt
├── .env.example
└── data/
    └── sample_queries.txt

notebooks/
└── Agentic_Research_Agent_Colab.ipynb   the runnable Colab demo
```

## Tech Stack

- **LLM**: Google Gemini via the `google-genai` SDK (free tier, no OpenAI key required)
- **Search**: `ddgs` (free DuckDuckGo search wrapper)
- **Extraction**: `requests` + `beautifulsoup4`
- **Agent framework**: none — a lightweight, hand-written state + loop (~4 small files)
- **Runtime**: Google Colab (no Docker, no local server, no Node.js)

## Google Colab Setup

1. Open `notebooks/Agentic_Research_Agent_Colab.ipynb` in Google Colab.
2. Click the **key icon (🔑 Secrets)** in the left sidebar.
3. Add a new secret named exactly `GEMINI_API_KEY` and paste your key as the value. Toggle "Notebook access" on.
4. Run the cells from top to bottom.

The notebook never asks you to paste your key into a cell — it's loaded via:

```python
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
```

## Gemini API Setup

1. Get a free API key at [Google AI Studio](https://aistudio.google.com/apikey).
2. Add it to Colab Secrets as `GEMINI_API_KEY` (see above).
3. (Optional) Override the model by setting a `GEMINI_MODEL` env var / Colab Secret. The project defaults to `gemini-3.5-flash` but **Gemini model availability changes over time and by account/region** — if the default isn't available to you, change `GEMINI_MODEL` and nothing else needs to change.

## Running the Agent

Inside the notebook:

```python
agent = build_agent(GEMINI_API_KEY)
state = agent.run("What are the main components of modern AI agent architectures?")
```

Or from the generated project files (e.g. after cloning from GitHub):

```bash
pip install -r requirements.txt
export GEMINI_API_KEY="your-key"
python -c "from research_agent import build_agent; build_agent('your-key').run('your question')"
```

## Example Research Tasks

```
1. What are the main components of modern AI agent architectures?
2. Compare RAG-based systems with long-context LLM approaches.
3. What are the major challenges in deploying autonomous AI agents?
```

More in `data/sample_queries.txt`. The notebook also accepts a custom question.

## Sample Agent Trace

```
==================================================
🤖 AGENTIC RESEARCH AGENT
==================================================

🎯 Goal:
What are the main components of modern AI agent architectures?

🧠 Planning...
✓ Research plan created (4 sub-questions).

🔎 "core components of an AI agent system"
✓ 5 results found
📄 https://example.org/agent-architecture
✓ Evidence collected
...

🧠 Analyzing research gaps...
✓ Additional research required

🔎 "AI agent memory and tool use design patterns"
✓ 4 results found
...

🧠 Synthesizing final report...

==================================================
📊 FINAL RESEARCH REPORT
==================================================
## Executive Summary
...
## Sources
1. https://example.org/agent-architecture
...
--------------------------------------------------
Run summary: steps=4 | llm_calls=3 | search_calls=2 | sources=7
--------------------------------------------------
```

## Free-Tier Optimization

Everything is bounded, and bounds are configurable in `config.py` / via env vars:

| Limit | Default | Purpose |
|---|---|---|
| `MAX_AGENT_STEPS` | 5 | caps total loop iterations |
| `MAX_SEARCH_RESULTS` | 5 | results per search call |
| `MAX_SOURCES` | 8 | total sources kept per run |
| `MAX_PAGE_CHARS` | 12000 | characters kept per fetched page |

A typical run makes **2–4 Gemini calls** (plan, 1–2 analysis rounds, synthesis) and never sends a full raw webpage to Gemini — pages are cleaned, truncated, and reduced to a handful of key points by `extract_evidence()` *before* any LLM sees them. The loop always logs LLM calls, search calls, sources collected, and steps taken at the end of a run.

## Limitations

- Web search results are not guaranteed to be complete or accurate; the agent surfaces what it finds and flags conflicts, but does not fact-check sources against ground truth.
- `extract_evidence()` is a heuristic (keyword-overlap) extractor, not a semantic one — it can occasionally keep a less-relevant paragraph over a more relevant one.
- Free-tier Gemini and free DuckDuckGo search both have rate limits; heavy or rapid re-running may need a short pause.
- The agent reasons over search snippets and page text only — it does not verify claims against multiple independent primary sources unless the search results themselves surface them.

## Responsible AI

- The agent **never fabricates sources** — the final report is instructed to cite only URLs that were actually collected during the run.
- Evidence is kept explicitly separate from the model's own inference; the synthesis prompt is constrained to the evidence provided.
- Conflicting information found across sources is surfaced rather than silently resolved.
- Uncertainty and research limitations are always included in the final report, not just on request.

## Future Improvements

- Add a lightweight re-ranking step for sources before extraction.
- Support parallel (rate-limited) page fetches for faster runs.
- Add a semantic (embedding-based) evidence extractor as an optional upgrade path.
- Persist run history to disk for comparison across research sessions.
- Reuse this `llm.py` / tool-system / agent-loop pattern for Projects 2–4 (Coding Agent, Sales Agent, Recruiting Agent).

## Author

Built as Project 1 of a 4-project Agentic AI portfolio, demonstrating a hand-built agent loop (planning, tool use, state, gap detection, synthesis) without relying on an agent framework.
''')
print("Generated: README.md")

Generated: README.md


In [20]:
# Cell 20 — Generate requirements.txt
Path("requirements.txt").write_text(r'''google-genai>=1.0.0
ddgs>=9.0.0
requests>=2.31.0
beautifulsoup4>=4.12.0
''')
print("Generated: requirements.txt")

Generated: requirements.txt


In [21]:
# Cell 21 — Verify project structure
expected_files = [
    "README.md",
    "research_agent.py",
    "tools.py",
    "llm.py",
    "config.py",
    "tests.py",
    "requirements.txt",
    ".env.example",
    "data/sample_queries.txt",
]

print("Project structure check:\n")
all_present = True
for f in expected_files:
    exists = Path(f).exists()
    all_present &= exists
    print(f"  {'✓' if exists else '✗'} {f}")

print("\nAll expected files present!" if all_present else "\nSome files are missing — re-run the earlier generation cells.")

Project structure check:

  ✓ README.md
  ✓ research_agent.py
  ✓ tools.py
  ✓ llm.py
  ✓ config.py
  ✓ tests.py
  ✓ requirements.txt
  ✓ .env.example
  ✓ data/sample_queries.txt

All expected files present!


In [22]:
# Cell 22 — Create ZIP for GitHub upload
PROJECT_DIR = Path("01-research-agent")
PROJECT_DIR.mkdir(exist_ok=True)
(PROJECT_DIR / "data").mkdir(exist_ok=True)

for f in expected_files:
    src = Path(f)
    dest = PROJECT_DIR / f
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(src.read_text())

zip_path = Path("agentic-research-agent.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in PROJECT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(PROJECT_DIR.parent))

print(f"Created {zip_path} ({zip_path.stat().st_size} bytes)")

Created agentic-research-agent.zip (19890 bytes)


In [23]:
# Cell 23 — Download ZIP
from google.colab import files
files.download(str(zip_path))

print("Download started. Unzip locally and push '01-research-agent/' to a new GitHub repo.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Unzip locally and push '01-research-agent/' to a new GitHub repo.
